# Unit Test Notebook: ASP Causal Study

Load results from either `golden_notebook.ipynb` or `agent_notebook.ipynb`
and run all 18 unit tests.

**Steps:**
1. Run `golden_notebook.ipynb` or `agent_notebook.ipynb` to generate the pkl file
2. Upload the pkl file to this session if using Colab
3. Set `PKL_FILE` in the next cell to the correct filename
4. Run All


In [ ]:
!pip install -r requirements.txt -q
print('Libraries ready.')

In [ ]:
import pickle, numpy as np, pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

PKL_FILE = 'golden_results.pkl'  # change to 'agent_results.pkl' to validate the agent

with open(PKL_FILE, 'rb') as f:
    _results = pickle.load(f)

globals().update(_results)

print(f"Loaded from: {PKL_FILE}")
print(f"Variables  : {list(_results.keys())}")
print(f"df shape   : {df.shape}")


## Unit Tests

In [ ]:
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

print("Running unit tests...\n")

# Section A: Survival Analysis

# A1: Dataset has required survival columns and valid values
assert 'time_to_improvement' in df.columns,  "A1 FAIL: time_to_improvement missing"
assert 'event_observed' in df.columns,        "A1 FAIL: event_observed missing"
assert df['event_observed'].isin([0, 1]).all(),"A1 FAIL: event_observed must be binary"
assert (df['time_to_improvement'] > 0).all(),  "A1 FAIL: survival times must be positive"
print("A1 PASS: Survival columns valid")

# A2: Censoring rate is in realistic range (10-60%)
censoring_rate = 1 - df['event_observed'].mean()
assert 0.10 < censoring_rate < 0.60,     f"A2 FAIL: censoring rate {censoring_rate:.2%} outside 10-60%"
print(f"A2 PASS: Censoring rate = {censoring_rate:.1%}")

# A3: At t=22, naive KM shows lower S(t) than correct KM
_kmf_naive   = KaplanMeierFitter().fit(df['time_to_improvement'],
                                       event_observed=np.ones(len(df)))
_kmf_correct = KaplanMeierFitter().fit(df['time_to_improvement'],
                                       event_observed=df['event_observed'])
_s_naive   = float(_kmf_naive.survival_function_at_times([22]).values[0])
_s_correct = float(_kmf_correct.survival_function_at_times([22]).values[0])
assert _s_correct >= _s_naive,     f"A3 FAIL: S_correct(22)={_s_correct:.4f} should be >= S_naive(22)={_s_naive:.4f}"
print(f"A3 PASS: S(22) naive={_s_naive:.4f}  correct={_s_correct:.4f}")

# A4: ASP significantly accelerates clinical improvement (log-rank p < 0.05)
_asp_mask    = df['asp_active'] == 1
_noasp_mask  = df['asp_active'] == 0
_lr = logrank_test(df.loc[_asp_mask,  'time_to_improvement'],
                   df.loc[_noasp_mask,'time_to_improvement'],
                   event_observed_A=df.loc[_asp_mask,  'event_observed'],
                   event_observed_B=df.loc[_noasp_mask,'event_observed'])
assert _lr.p_value < 0.05, f"A4 FAIL: log-rank p={_lr.p_value:.4f} not significant"
print(f"A4 PASS: Log-rank p = {_lr.p_value:.2e}")

# A5: Cox HR for ASP > 1 (ASP increases the rate of improvement, event = recovery)
assert 'cox_hr_asp' in dir() or 'cox_hr_asp' in globals(),     "A5 FAIL: cox_hr_asp not defined"
assert cox_hr_asp > 1.0, f"A5 FAIL: HR={cox_hr_asp:.3f} should be > 1 (ASP speeds recovery)"
assert cox_hr_asp < 10.0, f"A5 FAIL: HR={cox_hr_asp:.3f} implausibly high"
print(f"A5 PASS: Cox ASP HR = {cox_hr_asp:.4f}")

# A6: Proportional hazards assumption was formally tested
assert ph_test_performed == True,     "A6 FAIL: ph_test_performed must be True (Schoenfeld residual test required)"
print("A6 PASS: PH assumption tested")

# Section B: Difference-in-Differences

# B1: Hospital-year panel has 60 rows (15 hospitals x 4 years)
assert did_panel.shape[0] == 60,     f"B1 FAIL: DiD panel has {did_panel.shape[0]} rows, expected 60"
print(f"B1 PASS: Panel shape = {did_panel.shape}")

# B2: Parallel pre-trends hold (interaction coeff small, p > 0.05)
assert abs(pre_trend_coeff) < 1.0,     f"B2 FAIL: Pre-trend coeff={pre_trend_coeff:.4f} is too large"
assert pre_trend_pvalue > 0.05,     f"B2 FAIL: Pre-trend p={pre_trend_pvalue:.4f} < 0.05; parallel trends violated"
print(f"B2 PASS: Pre-trend coeff={pre_trend_coeff:.4f}, p={pre_trend_pvalue:.3f}")

# B3 and B4: ATT is negative and in plausible range
assert did_att_estimate < 0,     f"B3 FAIL: ATT={did_att_estimate:.3f} should be negative (ASP reduces duration)"
assert -6.0 < did_att_estimate < -0.5,     f"B4 FAIL: ATT={did_att_estimate:.3f} outside plausible range (-6, -0.5) days"
print(f"B3/B4 PASS: ATT = {did_att_estimate:.4f} days")

# B5: Staggered adoption timing was explicitly accounted for
assert staggered_did_used == True,     "B5 FAIL: staggered_did_used must be True"
print("B5 PASS: Staggered DiD used")

# Section C: Heterogeneous Treatment Effects

# C1: CATE vector covers all patients with no NaNs
assert len(cate_estimates) == len(df),     f"C1 FAIL: cate_estimates length {len(cate_estimates)} != df rows {len(df)}"
assert not np.any(np.isnan(cate_estimates)), "C1 FAIL: NaN in CATE estimates"
print(f"C1 PASS: CATE estimated for all {len(cate_estimates):,} patients")

# C2: CATE shows meaningful heterogeneity (not all identical)
_cate_std = np.std(cate_estimates)
assert _cate_std > 0.3,     f"C2 FAIL: CATE std={_cate_std:.4f} too low"
print(f"C2 PASS: CATE std = {_cate_std:.4f}")

# C3: Elderly (65+) benefit more from ASP than young (<40)
_cate_elderly = np.mean(cate_estimates[df['age_group'].values == '65+'])
_cate_young   = np.mean(cate_estimates[df['age_group'].values == '<40'])
assert _cate_elderly < _cate_young,     (f"C3 FAIL: Elderly CATE={_cate_elderly:.3f} should be more negative "
     f"than young CATE={_cate_young:.3f}")
print(f"C3 PASS: CATE elderly={_cate_elderly:.3f}, young={_cate_young:.3f}")

# C4: Mean CATE for treated aligns with DiD ATT (within factor of 3)
_ate_from_cate = float(np.mean(cate_estimates[df['asp_active'].values == 1]))
assert -8.0 < _ate_from_cate < -0.3,     f"C4 FAIL: Mean CATE for treated={_ate_from_cate:.3f} implausible"
print(f"C4 PASS: Mean CATE (treated) = {_ate_from_cate:.4f} days")

# C5: A proper meta-learner was used
assert meta_learner_used in ('T-learner', 'X-learner', 'both'),     f"C5 FAIL: meta_learner_used='{meta_learner_used}'"
print(f"C5 PASS: Meta-learner used = '{meta_learner_used}'")

# Section D: Mixed Effects Modeling

# D1: ICC is non-trivial
assert icc_value is not None, "D1 FAIL: icc_value not defined"
assert 0.02 < icc_value < 0.50,     f"D1 FAIL: ICC={icc_value:.4f} outside plausible 0.02-0.50 range"
print(f"D1 PASS: ICC = {icc_value:.4f}")

# D2: Mixed model outperforms naive OLS (lower AIC)
assert mixed_model_aic < naive_ols_aic,     (f"D2 FAIL: Mixed AIC={mixed_model_aic:.1f} should be < "
     f"Naive AIC={naive_ols_aic:.1f}")
print(f"D2 PASS: Mixed AIC={mixed_model_aic:.1f} < Naive AIC={naive_ols_aic:.1f}")

# D3: ASP reduces antibiotic duration in mixed effects model
assert mixed_asp_coeff < 0,     f"D3 FAIL: Mixed ASP coeff={mixed_asp_coeff:.4f} should be negative"
print(f"D3 PASS: Mixed effects ASP coeff = {mixed_asp_coeff:.4f}")

# D4: Hospital-level random effect variance is non-trivial
assert random_effect_variance > 0.05,     f"D4 FAIL: RE variance={random_effect_variance:.4f} too small"
print(f"D4 PASS: RE variance = {random_effect_variance:.4f}")

# D5: Cluster-robust SE used in Cox model
assert cluster_robust_se_used == True,     "D5 FAIL: cluster_robust_se_used must be True"
print("D5 PASS: Cluster-robust SE used in Cox model")

print("\n" + "="*60)
print("ALL 18 UNIT TESTS PASSED")
print("="*60)
